In [1]:
import json
import math
import time
from glob import glob
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import yaml
from shapely.geometry.base import BaseGeometry


# -----------------------------
# Config + utilities
# -----------------------------
def load_config(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def clamp_xpv(xpv_ha: float) -> float:
    if xpv_ha < 2 or xpv_ha > 100:
        raise ValueError(f"XPV requested_area_ha must be in [2, 100]. Got {xpv_ha}")
    return xpv_ha

def ha_to_m2(ha: float) -> float:
    return float(ha) * 10000.0

def panel_area_m2(cfg) -> float:
    return float(cfg["panel"]["length_m"]) * float(cfg["panel"]["width_m"])

def guess_nuts_layer(gpkg_path: str, preferred=None):
    layers = gpd.list_layers(gpkg_path)
    if preferred:
        return preferred
    return layers.iloc[0]["name"] if len(layers) else None

def county_name_from_row(row) -> str:
    for col in ["NAME_LATN", "NAME_ENGL", "NAME", "NUTS_NAME"]:
        if col in row and pd.notna(row[col]):
            return str(row[col])
    return str(row["NUTS_ID"])

def load_counties(cfg) -> gpd.GeoDataFrame:
    gpkg_path = cfg["project"]["nuts_gpkg_path"]
    layer = guess_nuts_layer(gpkg_path, cfg["project"].get("nuts_layer"))
    if not layer:
        raise RuntimeError(f"No layers found in {gpkg_path}")

    gdf = gpd.read_file(gpkg_path, layer=layer)

    if "CNTR_CODE" in gdf.columns:
        gdf = gdf[gdf["CNTR_CODE"] == cfg["project"]["country"]].copy()
    if "LEVL_CODE" in gdf.columns:
        gdf = gdf[gdf["LEVL_CODE"] == cfg["project"]["nuts_level"]].copy()

    if "NUTS_ID" not in gdf.columns:
        raise RuntimeError("Expected column 'NUTS_ID' in GeoPackage layer.")

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf

def find_substation_files(cfg):
    explicit = cfg["grid_proxy"].get("substations_geojson")
    if explicit and Path(explicit).exists():
        return [explicit]
    pattern = cfg["grid_proxy"].get("substations_glob")
    if not pattern:
        return []
    return sorted(glob(pattern))

def sanitize_points(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.empty:
        return gdf
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geometry.apply(lambda x: isinstance(x, BaseGeometry))].copy()
    gdf["geometry"] = gdf.geometry.apply(
        lambda geom: geom if geom.geom_type == "Point" else geom.representative_point()
    )
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf

def load_substations(cfg) -> gpd.GeoDataFrame:
    files = find_substation_files(cfg)
    if not files:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    frames = []
    for fp in files:
        try:
            g = gpd.read_file(fp)
            if g.crs is None:
                g = g.set_crs("EPSG:4326")
            g = sanitize_points(g)
            frames.append(g[["geometry"]].copy())
        except Exception:
            continue

    if not frames:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    out = pd.concat(frames, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=frames[0].crs)
    out = sanitize_points(out)
    return out

def load_cache(cache_path: Path) -> dict:
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    return {}

def save_cache(cache_path: Path, cache: dict):
    cache_path.write_text(json.dumps(cache, indent=2), encoding="utf-8")


# -----------------------------
# Land map (CSV or config fallback)
# -----------------------------
def load_land_map(cfg) -> dict:
    """
    Prefer outputs/available_land_by_nuts3.csv (if you have CORINE pipeline),
    otherwise fallback to config.yaml counties: { NUTS_ID: { available_ha: ... } }.
    """
    csv_path = Path(cfg["land"]["available_land_csv"])
    if csv_path.exists():
        land_df = pd.read_csv(csv_path)
        if "NUTS_ID" not in land_df.columns or "available_ha" not in land_df.columns:
            raise ValueError(f"{csv_path} must have columns: NUTS_ID, available_ha")
        return dict(zip(land_df["NUTS_ID"], land_df["available_ha"]))

    # fallback: config counties
    counties_cfg = cfg.get("counties", {}) or {}
    out = {}
    for nuts_id, rec in counties_cfg.items():
        if isinstance(rec, dict) and "available_ha" in rec:
            out[nuts_id] = float(rec["available_ha"])
    return out


# -----------------------------
# Feature loading (cloud + albedo)
# -----------------------------
def load_feature_table(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    if "NUTS_ID" not in df.columns:
        return pd.DataFrame()
    return df

def normalize_fraction(x):
    """
    Accept either [0..1] or [0..100] and normalize to [0..1].
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    x = float(x)
    if x > 1.5:  # assume percent
        x = x / 100.0
    return float(np.clip(x, 0.0, 1.0))

def build_country_month_means(feat_df: pd.DataFrame, prefix: str):
    """
    Returns dict month->mean value (normalized 0..1 where applicable).
    prefix: 'cloud' or 'albedo'
    """
    if feat_df.empty:
        return {m: np.nan for m in range(1, 13)}

    means = {}
    for m in range(1, 13):
        col = f"{prefix}_m{m:02d}"
        if col not in feat_df.columns:
            means[m] = np.nan
            continue
        vals = feat_df[col].apply(normalize_fraction if prefix == "cloud" else normalize_fraction)
        means[m] = float(np.nanmean(vals.values)) if np.isfinite(vals.values).any() else np.nan
    return means


# -----------------------------
# PVGIS calls (PVcalc -> annual + monthly)
# -----------------------------
def pvgis_pvcalc(lat: float, lon: float, cfg, session: requests.Session, cache: dict):
    """
    Returns (Ey_kWh_per_kWp, monthly_kWh_per_kWp[12]) for fixed system from PVcalc.
    """
    sys = cfg["pv_model"]["system"]
    base = cfg["pv_model"]["api_base"].rstrip("/")
    url = f"{base}/PVcalc"

    # include key params so cache is safe if you change config
    key = (
        f"{lat:.5f},{lon:.5f}|pp=1|loss={sys['loss_percent']}|tilt={sys['tilt_deg']}|az={sys['azimuth_deg']}"
        f"|tech={sys['pvtechchoice']}|mount={sys['mountingplace']}"
    )

    if key in cache:
        return cache[key]["Ey"], cache[key]["Em"]

    params = {
        "lat": lat,
        "lon": lon,
        "peakpower": 1.0,
        "loss": sys["loss_percent"],
        "angle": sys["tilt_deg"],
        "aspect": sys["azimuth_deg"],
        "pvtechchoice": sys["pvtechchoice"],
        "mountingplace": sys["mountingplace"],
        "outputformat": "json",
    }

    r = session.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    Ey = float(data["outputs"]["totals"]["fixed"]["E_y"])

    # Monthly extraction (robust)
    Em = [np.nan] * 12
    try:
        monthly = data["outputs"]["monthly"]["fixed"]
        # PVGIS returns list of dicts with 'month' and 'E_m' (common pattern)
        for rec in monthly:
            m = int(rec.get("month"))
            Em[m - 1] = float(rec.get("E_m"))
    except Exception:
        # fallback: no monthly available
        Em = [np.nan] * 12

    cache[key] = {"Ey": Ey, "Em": Em}
    return Ey, Em


# -----------------------------
# Satellite correction
# -----------------------------
def corrected_annual_from_satellite(
    Em_pvgis, cloud_row, albedo_row,
    cloud_country_means, albedo_country_means,
    gamma_albedo=0.05,
    clamp_cloud=(0.80, 1.20),
    clamp_albedo=(0.95, 1.05),
):
    """
    Em_pvgis: 12 monthly kWh/kWp
    cloud_row/albedo_row: pandas Series for county (may be None)
    country_means: dict month->mean
    """

    # If PVGIS monthly is missing, we can’t do month-wise correction reliably
    if Em_pvgis is None or not np.isfinite(np.array(Em_pvgis, dtype=float)).any():
        return np.nan, [np.nan]*12, [1.0]*12, [1.0]*12

    Em_corr = []
    f_cloud_list = []
    f_alb_list = []

    for m in range(1, 13):
        base = Em_pvgis[m-1]
        if not np.isfinite(base):
            Em_corr.append(np.nan)
            f_cloud_list.append(1.0)
            f_alb_list.append(1.0)
            continue

        # --- cloud factor: ratio of clear-sky fraction to country mean clear-sky fraction
        f_cloud = 1.0
        if cloud_row is not None:
            col = f"cloud_m{m:02d}"
            if col in cloud_row.index and pd.notna(cloud_row[col]):
                cloud = normalize_fraction(cloud_row[col])
                clear = 1.0 - cloud
                cloud_mu = cloud_country_means.get(m, np.nan)
                clear_mu = 1.0 - cloud_mu if np.isfinite(cloud_mu) else np.nan
                if np.isfinite(clear_mu) and clear_mu > 0.02:
                    f_cloud = float(clear / clear_mu)
        f_cloud = float(np.clip(f_cloud, clamp_cloud[0], clamp_cloud[1]))

        # --- albedo factor: very small adjustment (proxy effect unless bifacial modeled)
        f_alb = 1.0
        if albedo_row is not None:
            col = f"albedo_m{m:02d}"
            if col in albedo_row.index and pd.notna(albedo_row[col]):
                alb = normalize_fraction(albedo_row[col])
                alb_mu = albedo_country_means.get(m, np.nan)
                if np.isfinite(alb_mu):
                    f_alb = 1.0 + gamma_albedo * float(alb - alb_mu)
        f_alb = float(np.clip(f_alb, clamp_albedo[0], clamp_albedo[1]))

        Em_corr.append(base * f_cloud * f_alb)
        f_cloud_list.append(f_cloud)
        f_alb_list.append(f_alb)

    Ey_corr = float(np.nansum(np.array(Em_corr, dtype=float)))
    return Ey_corr, Em_corr, f_cloud_list, f_alb_list


# -----------------------------
# Grid proxy metrics
# -----------------------------
def brute_grid_metrics(rep_point, substation_points, radius_m: float):
    if not substation_points:
        return float("inf"), 0
    dists = [rep_point.distance(p) for p in substation_points]
    d_min = min(dists)
    cnt = sum(1 for d in dists if d <= radius_m)
    return d_min, cnt

def print_block(title: str, df: pd.DataFrame, cols: list[str]):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    if df.empty:
        print("(none)")
        return
    print(df[cols].to_string(index=False))


# -----------------------------
# Main
# -----------------------------
def main():
    cfg = load_config("config.yaml")
    debug = bool(cfg.get("debug", False))

    land_map = load_land_map(cfg)

    # Load feature CSVs
    cloud_df = load_feature_table(cfg["features"]["cloud_csv"])
    albedo_df = load_feature_table(cfg["features"]["albedo_csv"])

    cloud_country_means = build_country_month_means(cloud_df, "cloud")
    albedo_country_means = build_country_month_means(albedo_df, "albedo")

    # index feature rows by NUTS_ID
    cloud_df = cloud_df.set_index("NUTS_ID") if not cloud_df.empty else cloud_df
    albedo_df = albedo_df.set_index("NUTS_ID") if not albedo_df.empty else albedo_df

    cache_dir = ensure_dir(Path(cfg["run"]["cache_dir"]))
    cache_path = cache_dir / cfg["run"]["pvgis_cache_file"]
    cache = load_cache(cache_path)

    xpv = clamp_xpv(float(cfg["xpv"]["requested_area_ha"]))

    counties = load_counties(cfg)
    substations = load_substations(cfg)

    proj_crs = cfg["project"]["crs_area_distance"]
    counties_proj = counties.to_crs(proj_crs)
    substations_proj = substations.to_crs(proj_crs)
    sub_pts = substations_proj.geometry.tolist()

    if debug:
        print(f"[DEBUG] counties: {len(counties_proj)} CRS={counties_proj.crs}")
        print(f"[DEBUG] substations: {len(substations_proj)} CRS={substations_proj.crs}")
        print(f"[DEBUG] land_map entries: {len(land_map)}")
        print(f"[DEBUG] cloud features: {0 if cloud_df.empty else len(cloud_df)}")
        print(f"[DEBUG] albedo features: {0 if albedo_df.empty else len(albedo_df)}")

    grid_cfg = cfg["constraints"]["grid"]
    require_grid = bool(grid_cfg.get("require_grid", True))
    max_km = float(grid_cfg["nearest_substation_max_km"])
    radius_km = float(grid_cfg["radius_km"])
    k_min = int(grid_cfg["min_substations_within_radius"])
    radius_m = radius_km * 1000.0

    session = requests.Session()
    sleep_s = float(cfg["run"].get("request_sleep_s", 0.0))

    # Correction strength (keep small unless you have training labels)
    gamma_albedo = float(cfg.get("satellite_correction", {}).get("gamma_albedo", 0.05))

    results = []

    for _, row in counties_proj.iterrows():
        county_id = row["NUTS_ID"]
        county_name = county_name_from_row(row)

        available_ha = float(land_map.get(county_id, 0.0))
        land_ok = available_ha >= xpv

        rep = row.geometry.representative_point()
        rep_wgs = gpd.GeoSeries([rep], crs=counties_proj.crs).to_crs("EPSG:4326").iloc[0]
        lat, lon = float(rep_wgs.y), float(rep_wgs.x)

        on_fail = cfg["pv_model"].get("on_fail", "fallback")
        fallback = float(cfg["pv_model"].get("fallback_Ey_kWh_per_kWp", 1300))

        error = None
        try:
            Ey_pvgis, Em_pvgis = pvgis_pvcalc(lat, lon, cfg, session, cache)
        except Exception as e:
            if on_fail == "skip":
                Ey_pvgis, Em_pvgis = float("nan"), [np.nan]*12
                error = f"PVGIS failed: {e}"
            else:
                Ey_pvgis, Em_pvgis = fallback, [np.nan]*12
                error = f"PVGIS failed (fallback used): {e}"

        # Feature rows (may be missing)
        cloud_row = cloud_df.loc[county_id] if (not cloud_df.empty and county_id in cloud_df.index) else None
        albedo_row = albedo_df.loc[county_id] if (not albedo_df.empty and county_id in albedo_df.index) else None

        Ey_corr, Em_corr, f_cloud_m, f_alb_m = corrected_annual_from_satellite(
            Em_pvgis,
            cloud_row,
            albedo_row,
            cloud_country_means,
            albedo_country_means,
            gamma_albedo=gamma_albedo
        )

        # If no monthly PVGIS, fallback: use annual PVGIS uncorrected
        Ey_used = Ey_corr if np.isfinite(Ey_corr) and Ey_corr > 0 else Ey_pvgis

        panel_kwp = float(cfg["panel"]["p_stc_kwp"])
        gcr = float(cfg["panel"]["ground_coverage_ratio"])
        a_panel = panel_area_m2(cfg)

        eff_ha = min(xpv, available_ha)
        n_panels = math.floor((ha_to_m2(eff_ha) * gcr) / a_panel) if eff_ha > 0 else 0

        annual_energy_kWh = Ey_used * panel_kwp * n_panels

        d_m, cnt = brute_grid_metrics(rep, sub_pts, radius_m)
        d_km = d_m / 1000.0 if math.isfinite(d_m) else float("inf")

        grid_ok = True
        if require_grid:
            grid_ok = (d_km <= max_km) and (cnt >= k_min)

        eligible = land_ok and grid_ok

        results.append({
            "county_id": county_id,
            "county_name": county_name,

            "available_ha": available_ha,
            "requested_xpv_ha": xpv,
            "effective_ha_used": eff_ha,
            "missing_ha_to_fit": max(xpv - available_ha, 0.0),
            "land_ok": land_ok,

            "grid_ok": grid_ok,
            "eligible": eligible,
            "nearest_substation_km": d_km if math.isfinite(d_m) else None,
            "substations_within_radius": int(cnt),

            "lat": lat,
            "lon": lon,

            # PVGIS base + corrected
            "pvgis_Ey_kWh_per_kWp": float(Ey_pvgis) if np.isfinite(Ey_pvgis) else None,
            "sat_corrected_Ey_kWh_per_kWp": float(Ey_corr) if np.isfinite(Ey_corr) else None,
            "Ey_used_kWh_per_kWp": float(Ey_used) if np.isfinite(Ey_used) else None,

            "panel_kWp": panel_kwp,
            "n_panels_used": int(n_panels),
            "annual_energy_kWh": float(annual_energy_kWh),

            # feature summaries (optional)
            "cloud_mean": float(normalize_fraction(cloud_row["cloud_mean"])) if cloud_row is not None and "cloud_mean" in cloud_row.index else None,
            "albedo_mean": float(normalize_fraction(albedo_row["albedo_mean"])) if albedo_row is not None and "albedo_mean" in albedo_row.index else None,

            "error": error,
        })

        if sleep_s > 0:
            time.sleep(sleep_s)

    save_cache(cache_path, cache)

    df = pd.DataFrame(results)

    df_eligible = df[df["eligible"] == True].copy()
    df_eligible = df_eligible.sort_values(
        by=["annual_energy_kWh", "Ey_used_kWh_per_kWp", "nearest_substation_km", "substations_within_radius"],
        ascending=[False, False, True, False],
        na_position="last",
    )

    df_nofit_land = df[df["available_ha"] < xpv].copy()
    df_nofit_land = df_nofit_land.sort_values(
        by=["annual_energy_kWh", "grid_ok", "Ey_used_kWh_per_kWp", "nearest_substation_km"],
        ascending=[False, False, False, True],
        na_position="last",
    )

    out_csv = Path(cfg["run"]["output_csv"])
    out_json = Path(cfg["run"]["output_json"])
    ensure_dir(out_csv.parent)
    ensure_dir(out_json.parent)

    df.to_csv(out_csv, index=False, encoding="utf-8")
    df.to_json(out_json, orient="records", indent=2, force_ascii=False)

    top_n = int(cfg["run"]["top_n"])
    extra_nn = int(cfg["run"]["extra_nn"])

    err_rows = df["error"].notna().sum()
    if err_rows > 0:
        print(f"\n{err_rows}/{len(df)} rows had PVGIS errors (fallback may have been used).")
        print(df.loc[df["error"].notna(), ["county_id", "county_name", "error"]].head(3).to_string(index=False))

    if df_eligible.empty:
        max_km = float(cfg["constraints"]["grid"]["nearest_substation_max_km"])
        radius_km = float(cfg["constraints"]["grid"]["radius_km"])
        k_min = int(cfg["constraints"]["grid"]["min_substations_within_radius"])

        print(f"\nNo counties satisfy BOTH hard constraints for XPV={xpv:.1f} ha:")
        print(f"   - land_ok: available_ha >= {xpv:.1f}")
        print(f"   - grid_ok: nearest_substation_km <= {max_km} AND substations within {radius_km}km >= {k_min}")
        print(f"\nShowing {extra_nn} best alternatives if you needed smaller land (available_ha < XPV):\n")

        show = df_nofit_land.head(extra_nn)
        print_block(
            f"Best {extra_nn} smaller-land options (ranked by total annual energy achievable)",
            show,
            cols=[
                "county_id", "county_name",
                "available_ha", "missing_ha_to_fit",
                "grid_ok", "nearest_substation_km", "substations_within_radius",
                "n_panels_used", "annual_energy_kWh"
            ],
        )
    else:
        top = df_eligible.head(top_n)
        print_block(
            f"Top {top_n} counties (HARD constraints satisfied) — ranked by TOTAL annual energy (sat-corrected PVGIS)",
            top,
            cols=[
                "county_id", "county_name",
                "available_ha",
                "nearest_substation_km", "substations_within_radius",
                "n_panels_used", "annual_energy_kWh",
                "Ey_used_kWh_per_kWp"
            ],
        )

        extra = df_nofit_land.head(extra_nn)
        print(f"\nNote: The counties below do NOT fit XPV={xpv:.1f} ha by land,")
        print("but are shown as strong candidates if you needed a smaller area. Each shows exact available land.\n")

        print_block(
            f"Extra {extra_nn} smaller-land options (available_ha < XPV) — ranked by achievable annual energy",
            extra,
            cols=[
                "county_id", "county_name",
                "available_ha", "missing_ha_to_fit",
                "grid_ok", "nearest_substation_km", "substations_within_radius",
                "n_panels_used", "annual_energy_kWh",
                "Ey_used_kWh_per_kWp"
            ],
        )

    print("\nSaved outputs:")
    print(f" - {out_csv.resolve()}")
    print(f" - {out_json.resolve()}")
    print(f" - PVGIS cache: {cache_path.resolve()}")


if __name__ == "__main__":
    main()


Top 8 counties (HARD constraints satisfied) — ranked by TOTAL annual energy (sat-corrected PVGIS)
county_id county_name  available_ha  nearest_substation_km  substations_within_radius  n_panels_used  annual_energy_kWh  Ey_used_kWh_per_kWp
    RO223   Constanţa      556336.0               4.054993                         89         122448       8.160218e+07          1480.940243
    RO315    Ialomiţa      381849.0               6.611009                          3         122448       8.100545e+07          1470.110596
    RO411        Dolj      577548.0              10.567411                         15         122448       7.826565e+07          1420.387900
    RO414         Olt      436624.0               3.735402                         15         122448       7.739322e+07          1404.554877
    RO312    Călăraşi      440232.0               9.205459                          8         122448       7.612372e+07          1381.515682
    RO314     Giurgiu      273272.0               6.529